# 📸 Image → 3D Model

Upload a photo of any object and get a 3D model (`.obj` file) you can download.

**Powered by:** [TripoSR](https://github.com/VAST-AI-Research/TripoSR) by Stability AI

### Tips for best results:
- Use a clear photo with the object centered
- Plain/white background works best
- Good lighting, no shadows
- One object per image

> **Requires:** Run `01_setup.ipynb` first.

In [ ]:
# Make sure TripoSR is installed
import os
if not os.path.exists('/content/TripoSR'):
    !git clone https://github.com/VAST-AI-Research/TripoSR /content/TripoSR
    %cd /content/TripoSR
    !pip install -q -r requirements.txt
else:
    %cd /content/TripoSR

print('TripoSR ready.')

In [ ]:
# Import libraries
import torch
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display as ipy_display
from google.colab import files
import shutil

print(f'GPU available: {torch.cuda.is_available()}')
print(f'Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU (slower)"}')

In [ ]:
# 📤 Upload your image
print('Select an image file from your computer:')
uploaded = files.upload()

image_filename = list(uploaded.keys())[0]
image_path = f'/content/{image_filename}'

# Preview the uploaded image
img = Image.open(image_path)
print(f'Image loaded: {image_filename} ({img.size[0]}x{img.size[1]} px)')
plt.figure(figsize=(5, 5))
plt.imshow(img)
plt.axis('off')
plt.title('Your uploaded image')
plt.show()

In [ ]:
# Remove background (optional but improves 3D quality significantly)
!pip install -q rembg
from rembg import remove

print('Removing background...')
with open(image_path, 'rb') as i:
    with open('/content/input_nobg.png', 'wb') as o:
        o.write(remove(i.read()))

# Show before/after
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(Image.open(image_path))
axes[0].set_title('Original')
axes[0].axis('off')
axes[1].imshow(Image.open('/content/input_nobg.png'))
axes[1].set_title('Background Removed')
axes[1].axis('off')
plt.tight_layout()
plt.show()

final_input = '/content/input_nobg.png'
print('Background removed. Using cleaned image for 3D generation.')

In [ ]:
# 🏗️ Generate 3D model with TripoSR
output_dir = '/content/3d_output'
os.makedirs(output_dir, exist_ok=True)

print('Generating 3D model... (this takes ~1-2 minutes)')
print('The T4 GPU will do the heavy lifting.')

!python /content/TripoSR/run.py \
    {final_input} \
    --output-dir {output_dir} \
    --device cuda \
    --render

print('\n3D model generation complete!')

In [ ]:
# Find and list generated files
import glob

generated_files = glob.glob(f'{output_dir}/**/*', recursive=True)
obj_files = [f for f in generated_files if f.endswith('.obj')]
glb_files = [f for f in generated_files if f.endswith('.glb')]

print('Generated files:')
for f in generated_files:
    if os.path.isfile(f):
        size_kb = os.path.getsize(f) / 1024
        print(f'  {f} ({size_kb:.1f} KB)')

print(f'\nFound {len(obj_files)} .obj file(s) and {len(glb_files)} .glb file(s)')

In [ ]:
# Preview renders (if TripoSR generated them)
render_images = glob.glob(f'{output_dir}/**/*.png', recursive=True)

if render_images:
    print(f'Showing {len(render_images)} render previews:')
    cols = min(len(render_images), 4)
    fig, axes = plt.subplots(1, cols, figsize=(4 * cols, 4))
    if cols == 1:
        axes = [axes]
    for i, img_path in enumerate(render_images[:4]):
        axes[i].imshow(Image.open(img_path))
        axes[i].axis('off')
        axes[i].set_title(f'View {i+1}')
    plt.tight_layout()
    plt.show()
else:
    print('No render previews found. The .obj/.glb file is still generated.')

In [ ]:
# 📥 Download the 3D model files
download_files = obj_files + glb_files

if download_files:
    print('Downloading 3D model files...')
    for f in download_files:
        print(f'  Downloading: {os.path.basename(f)}')
        files.download(f)
    print('\nDone! Open .obj files in:')
    print('  - Windows: 3D Viewer (built-in)')
    print('  - Online: https://3dviewer.net (drag and drop)')
    print('  - Blender (free, professional)')
else:
    print('No .obj or .glb files found. Check the output above for errors.')